# 05-5. 데이터 검증 — 풀이 검증

## Goal

여러 오류를 구조화하고 원본 레코드를 변경하지 않는다.

> 학습자용 TODO를 먼저 완성한 뒤 참고한다.


## Setup

fixture와 실행 환경을 확인한다.


In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("requirements.txt가 있는 저장소 루트에서 JupyterLab을 실행하세요.")


ROOT = find_project_root()
FIXTURE_DIR = ROOT / "fixtures" / "05-text-processing"

assert sys.version_info >= (3, 10)
assert FIXTURE_DIR.is_dir()

print("Python:", sys.version.split()[0])
print("실습 데이터:", FIXTURE_DIR)


import json
import re
from ipaddress import ip_address, ip_network
from urllib.parse import unquote, urlsplit

fixture_path = FIXTURE_DIR / "validation-records.jsonl"
raw_lines = fixture_path.read_text(encoding="utf-8").splitlines()
records, json_errors = [], []
for line_number, line in enumerate(raw_lines, start=1):
    try:
        records.append(json.loads(line))
    except json.JSONDecodeError:
        encoded = line.encode("utf-8")
        json_errors.append({
            "line": line_number,
            "code": "INVALID_JSON",
            "byte_length": len(encoded),
        })


## Steps

참고 구현을 실행한다.


In [ ]:
ALLOWED_RESULTS = {"SUCCESS", "FAILURE", "LOCKED"}
ACCOUNT_PATTERN = re.compile(r"[A-Za-z0-9_-]{3,32}")
BAD_PERCENT = re.compile(r"%(?![0-9A-Fa-f]{2})")
ENCODED_AGAIN = re.compile(r"%[0-9A-Fa-f]{2}")


def normalize_request_path(target: str) -> str:
    if not isinstance(target, str) or not target or len(target) > 2048:
        raise ValueError("잘못된 target이다.")
    if any(character in target for character in ("\x00", "\r", "\n", "\\")):
        raise ValueError("허용되지 않은 문자가 있다.")
    try:
        parts = urlsplit(target)
    except ValueError as exc:
        raise ValueError("target 구조를 해석할 수 없다.") from exc
    if parts.scheme or parts.netloc or parts.fragment or not parts.path.startswith("/"):
        raise ValueError("origin-form 경로가 필요하다.")
    if BAD_PERCENT.search(parts.path):
        raise ValueError("잘못된 percent-encoding이다.")
    try:
        decoded = unquote(parts.path, encoding="utf-8", errors="strict")
    except UnicodeDecodeError as exc:
        raise ValueError("UTF-8 경로가 아니다.") from exc
    if ENCODED_AGAIN.search(decoded):
        raise ValueError("중첩 인코딩을 허용하지 않는다.")
    segments = []
    for segment in decoded.split("/"):
        if segment in ("", "."):
            continue
        if segment == "..":
            raise ValueError("상위 경로 이동을 허용하지 않는다.")
        segments.append(segment)
    return "/" + "/".join(segments)


def validate_account(value: object) -> str:
    if not isinstance(value, str):
        raise TypeError("계정은 문자열이어야 한다.")
    cleaned = value.strip()
    if ACCOUNT_PATTERN.fullmatch(cleaned) is None:
        raise ValueError("계정 형식이 올바르지 않다.")
    return cleaned


def validate_ip(value: object) -> str:
    if not isinstance(value, str):
        raise TypeError("IP는 문자열이어야 한다.")
    cleaned = value.strip()
    if "%" in cleaned:
        raise ValueError("영역 ID가 포함된 IPv6는 허용하지 않는다.")
    return str(ip_address(cleaned))


def validate_result(value: object) -> str:
    if not isinstance(value, str):
        raise TypeError("인증 결과는 문자열이어야 한다.")
    normalized = value.strip().upper()
    if normalized not in ALLOWED_RESULTS:
        raise ValueError("허용되지 않은 인증 결과다.")
    return normalized


def validate_risk_score(value: object) -> int:
    if isinstance(value, bool) or not isinstance(value, int):
        raise TypeError("위험 점수는 정수여야 한다.")
    if not 0 <= value <= 100:
        raise ValueError("위험 점수 범위를 벗어났다.")
    return value


def validate_record(record: dict) -> tuple[dict, list[dict]]:
    cleaned, errors = {}, []

    def add(field: str, code: str, message: str) -> None:
        errors.append({"field": field, "code": code, "message": message})

    if not isinstance(record, dict):
        return {}, [{"field": "record", "code": "INVALID_TYPE", "message": "객체가 필요하다."}]

    validators = {
        "account": validate_account,
        "ip": validate_ip,
        "result": validate_result,
        "risk_score": validate_risk_score,
        "target": normalize_request_path,
    }

    for field, validator in validators.items():
        if field not in record:
            add(field, "MISSING", "필수 필드가 없다.")
            continue
        try:
            cleaned[field] = validator(record[field])
        except (TypeError, ValueError):
            add(field, "INVALID", "필드 규칙을 만족하지 않는다.")

    return cleaned, errors


def make_shareable_record(record: dict) -> dict:
    parsed_ip = ip_address(record["ip"])
    prefix = 24 if parsed_ip.version == 4 else 64
    account = record["account"]
    return {
        "account": account[:2] + "*" * max(0, len(account) - 2),
        "ip_network": str(ip_network(f"{parsed_ip}/{prefix}", strict=False)),
        "result": record["result"],
        "risk_score": record["risk_score"],
        "path": record["target"],
    }


results = [validate_record(record) for record in records]
valid_records = [cleaned for cleaned, errors in results if not errors]
invalid_records = [errors for _, errors in results if errors]
shareable = [make_shareable_record(record) for record in valid_records]
print("정상:", len(valid_records), "검증 오류:", len(invalid_records), "JSON 오류:", len(json_errors))
shareable


## Checks

경계값과 fixture 결과를 대조한다.


In [ ]:
assert [len(errors) for _, errors in results] == [0, 5, 0, 2]
assert len(json_errors) == 1
assert len(raw_lines) == len(valid_records) + len(invalid_records) + len(json_errors)
before = records[0].copy()
cleaned, errors = validate_record(records[0])
assert records[0] == before and cleaned["result"] == "SUCCESS" and not errors
assert cleaned["target"] == "/login"
assert valid_records[1]["ip"] == "2001:db8::10"
assert all("ip" not in record and "account" in record for record in shareable)
assert all("token" not in record["path"] for record in shareable)
print("검증 통과")


## Next Steps

오류 코드는 표시 문구와 분리해 집계·다국어·재처리에 사용한다.
